In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Academic Research Crew

## What We're Building

A four-agent academic research workflow:

| Agent | Role |
|-------|------|
| Literature Searcher | Find relevant papers and source metadata |
| Summarizer | Produce structured paper summaries and synthesis |
| Gap Finder | Identify what the literature still misses |
| Bibliography Agent | Compile formatted references and final brief |

## Multi-Agent Research Flow

1. Literature Searcher gathers and organizes papers by theme.
2. Summarizer converts paper list into consistent evidence summaries.
3. Gap Finder inspects those summaries to surface methodological and domain gaps.
4. Bibliography Agent assembles executive summary, findings, gaps, and citations.

This design moves from retrieval -> interpretation -> critique -> publication-ready output.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Research Topic

In [3]:
research_topic = """
Topic: Retrieval-Augmented Generation (RAG) for domain-specific applications
Field: Natural Language Processing / Information Retrieval
Focus: How RAG systems perform when applied to specialized domains
like legal, medical, and financial text — compared to general-purpose LLMs.
Time range: 2022-2025
"""

print("Research topic loaded")

Research topic loaded


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    lit_searcher = Agent(
        role="Literature Searcher",
        goal="Find and catalog the most relevant academic papers on the research topic",
        backstory=(
            "You are a research librarian who specializes in finding high-signal papers. "
            "You prioritize relevance, recency, and venue quality."
        ),
        llm=llm,
        verbose=True,
    )

    summarizer = Agent(
        role="Summarizer",
        goal="Create concise and accurate summaries of academic papers",
        backstory=(
            "You are a research writer who extracts methods, findings, and limitations "
            "without overstating claims."
        ),
        llm=llm,
        verbose=True,
    )

    gap_finder = Agent(
        role="Gap Finder",
        goal="Identify unexplored research gaps and high-impact follow-up questions",
        backstory=(
            "You are a senior academic advisor skilled at finding what literature misses "
            "across methods, datasets, and deployment settings."
        ),
        llm=llm,
        verbose=True,
    )

    bibliography_agent = Agent(
        role="Bibliography Agent",
        goal="Compile the final research brief with structured and formatted references",
        backstory=(
            "You are a meticulous research editor who assembles publication-ready "
            "briefs and consistent citation sections."
        ),
        llm=llm,
        verbose=True,
    )
else:
    lit_searcher = {"role": "Literature Searcher"}
    summarizer = {"role": "Summarizer"}
    gap_finder = {"role": "Gap Finder"}
    bibliography_agent = {"role": "Bibliography Agent"}

print("4 agents created: Literature Searcher, Summarizer, Gap Finder, Bibliography Agent")


4 agents created: Literature Searcher, Summarizer, Gap Finder, Bibliography Agent


## Step 4 — Create Tasks

In [5]:
if CREWAI_AVAILABLE:
    search_task = Task(
        description=f"""Find the most relevant academic papers for this research topic:

{research_topic}

Find 8-10 papers and for each provide:
1. Title, authors, year, venue
2. Key contribution in one sentence
3. Why it is relevant to this topic
4. Estimated citation count

Organize papers by sub-theme.""",
        expected_output="A catalog of 8-10 relevant papers organized by theme.",
        agent=lit_searcher,
    )

    summary_task = Task(
        description="""For each paper found by the literature searcher, create structured summaries:
1. Problem
2. Method
3. Key results
4. Limitations
5. Relevance

End with cross-paper synthesis.""",
        expected_output="Detailed summaries of each paper plus a synthesis.",
        agent=summarizer,
        context=[search_task],
    )

    gap_task = Task(
        description="""Based on the summaries, identify:
1. Methodological gaps
2. Domain/application gaps
3. Real-world deployment gaps
4. Three concrete research questions
5. Feasible methods for each question""",
        expected_output="Research gaps analysis with 3 proposed research questions.",
        agent=gap_finder,
        context=[summary_task],
    )

    bib_task = Task(
        description="""Compile the final research brief:
1. Executive summary
2. Thematic bibliography in APA style
3. Key findings
4. Research gaps
5. Recommended reading order
6. Suggested next research steps""",
        expected_output="A complete research brief with formatted bibliography.",
        agent=bibliography_agent,
        context=[search_task, summary_task, gap_task],
        markdown=True,
    )
else:
    search_task = {"name": "Literature search task"}
    summary_task = {"name": "Summarization task"}
    gap_task = {"name": "Gap analysis task"}
    bib_task = {"name": "Bibliography task"}

print("4 tasks created with flow: search -> summarize -> gap analysis -> bibliography")


4 tasks created with flow: search -> summarize -> gap analysis -> bibliography


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Research brief generated with thematic bibliography and gap-driven questions."
        self.tasks_output = [
            _DemoTaskOutput("Literature Search", "8-10 papers grouped by foundational, domain, evaluation, and recent advances."),
            _DemoTaskOutput("Summaries", "Structured paper summaries plus cross-paper synthesis."),
            _DemoTaskOutput("Gap Analysis", "Methodological, domain, and deployment gaps with 3 research questions."),
            _DemoTaskOutput("Bibliography", "Final brief with APA-style thematic references and reading order."),
        ]


if CREWAI_AVAILABLE:
    research_crew = Crew(
        agents=[lit_searcher, summarizer, gap_finder, bibliography_agent],
        tasks=[search_task, summary_task, gap_task, bib_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Academic research crew assembled.")
    result = research_crew.kickoff()
    print("\nResearch brief complete.")
else:
    print("Demo mode: showing multi-agent research flow without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing multi-agent research flow without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("FINAL RESEARCH BRIEF")
print("=" * 60)
print(result.raw)

preview("Literature Search Output", result.tasks_output[0].raw)
preview("Summarizer Output", result.tasks_output[1].raw)
preview("Gap Finder Output", result.tasks_output[2].raw)
preview("Bibliography Output", result.tasks_output[3].raw)


FINAL RESEARCH BRIEF
Research brief generated with thematic bibliography and gap-driven questions.

Literature Search Output
8-10 papers grouped by foundational, domain, evaluation, and recent advances.

Summarizer Output
Structured paper summaries plus cross-paper synthesis.

Gap Finder Output
Methodological, domain, and deployment gaps with 3 research questions.

Bibliography Output
Final brief with APA-style thematic references and reading order.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Multi-agent flow | Literature Searcher -> Summarizer -> Gap Finder -> Bibliography Agent |
| Context handoff | Downstream tasks receive upstream outputs via `context=[...]` |
| Research critique | Gap Finder converts synthesis into testable next questions |
| Publication readiness | Bibliography Agent structures final brief and references |

## Extensions

- Integrate live search tools (Semantic Scholar API, OpenAlex)
- Add PDF extraction tools for direct paper ingestion
- Export bibliography to .bib and cite keys
- Add a reviewer agent for methodological risk scoring
